# Graph RAG on Raw Data (No Entity Resolution)

This notebook builds a **graph-augmented RAG** system directly on the raw Open Sanctions and Open Ownership datasets — no Senzing entity resolution involved. It combines the approaches from notebooks 05 (raw data vectorization) and 07 (graph-augmented RAG) to show what graph RAG looks like *before* entity resolution.

**What this notebook does:**
1. Loads the raw source records (Open Sanctions + Open Ownership)
2. Builds a NetworkX graph from the records and their declared relationships
3. Creates vector embeddings of each record and stores them in LanceDB
4. Implements a RAG pipeline that combines vector search with graph neighbor expansion

**Key limitation to observe:** Without entity resolution, the Open Sanctions and Open Ownership subgraphs are **completely disconnected** — the same real-world entity appears as separate, unlinked nodes across data sources. Compare results here with notebook 07 to see the difference ER makes.

In [1]:
import json
import os

import anthropic
import lancedb
import networkx as nx
import openai
import pyarrow as pa
from pyvis.network import Network
from sentence_transformers import SentenceTransformer
from IPython.display import display, HTML

print("All imports successful")

All imports successful


## Load Raw Data

Load the original Open Sanctions and Open Ownership JSON files (JSONL format — one JSON object per line). Deduplicate by `RECORD_ID` since Open Ownership contains duplicate lines.

In [2]:
DATA_DIR = "/workspace/data"
LANCEDB_PATH = "/workspace/lancedb_data"


def load_jsonl(filepath: str) -> list[dict]:
    """Load a JSONL file (one JSON object per line)."""
    records = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


sanctions_records = load_jsonl(os.path.join(DATA_DIR, "open-sanctions-v4.jsonl"))
ownership_records_raw = load_jsonl(os.path.join(DATA_DIR, "open-ownership-v4.jsonl"))

# Deduplicate Open Ownership by RECORD_ID (file contains duplicate lines)
seen_ids = set()
ownership_records = []
for r in ownership_records_raw:
    rid = r.get("RECORD_ID", "")
    if rid not in seen_ids:
        seen_ids.add(rid)
        ownership_records.append(r)

all_records = sanctions_records + ownership_records

print(f"Open Sanctions records: {len(sanctions_records)}")
print(f"Open Ownership records (deduplicated): {len(ownership_records)}")
print(f"Total raw records: {len(all_records)}")

Open Sanctions records: 24
Open Ownership records (deduplicated): 258
Total raw records: 282


## Build Graph from Raw Records

Construct a NetworkX graph directly from the raw data. Each record becomes a node, and edges are created by resolving `REL_POINTER_KEY` references in each record's `RELATIONSHIPS` array.

**Edge types:**
- **Directorship** — person directs an organization (Open Sanctions)
- **Shareholding** — entity owns shares in another, with ownership % range (Open Ownership)
- **Voting Rights** — entity holds voting rights in another (Open Ownership)
- **Board Appointment** — entity appoints board members of another (Open Ownership)
- **Family** — family relationship between persons (Open Sanctions)

**Important:** Open Sanctions uses domain `OPEN-SANCTIONS` and Open Ownership uses domain `OOR`. There are **no cross-domain relationship pointers**, so the two subgraphs will be completely disconnected.

In [3]:
def _get_features(record: dict) -> list[dict]:
    """Return the FEATURES list, falling back to an empty list."""
    return record.get("FEATURES", [])


def get_name(record: dict) -> str:
    """Extract the primary name from a record (v4 FEATURES format)."""
    for feat in _get_features(record):
        for key in ("NAME_FULL", "NAME_ORG", "PRIMARY_NAME_ORG", "PRIMARY_NAME_FULL"):
            if feat.get(key):
                return feat[key]
    if record.get("PRIMARY_NAME_FULL"):
        return record["PRIMARY_NAME_FULL"]
    return "Unknown"


def get_record_type(record: dict) -> str:
    """Extract RECORD_TYPE from the FEATURES list."""
    for feat in _get_features(record):
        if feat.get("RECORD_TYPE"):
            return feat["RECORD_TYPE"]
    return "UNKNOWN"


# Build lookup tables: map (domain, key) -> record
# Open Sanctions: domain "OPEN-SANCTIONS", key = RECORD_ID
# Open Ownership: domain "OOR", key = RECORD_ID
domain_map = {
    "OPEN-SANCTIONS": "OPEN-SANCTIONS",
    "OOR": "OPEN-OWNERSHIP",
}

record_by_id = {}  # (DATA_SOURCE, RECORD_ID) -> record
record_id_by_domain_key = {}  # (pointer_domain, pointer_key) -> (DATA_SOURCE, RECORD_ID)

for r in all_records:
    source = r.get("DATA_SOURCE", "")
    rid = r.get("RECORD_ID", "")
    record_by_id[(source, rid)] = r

    # Map this record so relationship pointers can find it
    if source == "OPEN-SANCTIONS":
        record_id_by_domain_key[("OPEN-SANCTIONS", rid)] = (source, rid)
    elif source == "OPEN-OWNERSHIP":
        record_id_by_domain_key[("OOR", rid)] = (source, rid)

print(f"Records indexed: {len(record_by_id)}")
print(f"Domain-key mappings: {len(record_id_by_domain_key)}")

Records indexed: 282
Domain-key mappings: 282


In [4]:
G = nx.DiGraph()

# Step 1: Add all records as nodes
for (source, rid), record in record_by_id.items():
    node_id = f"{source}_{rid}"
    name = get_name(record)
    record_type = get_record_type(record)

    # Collect risk topics from top-level field
    risk_topics = record.get("risk_topics", "")
    risks = [t.strip() for t in risk_topics.split(",") if t.strip()] if risk_topics else []

    G.add_node(
        node_id,
        name=name,
        record_type=record_type,
        data_source=source,
        record_id=rid,
        risks=risks,
    )

print(f"Nodes added: {G.number_of_nodes()}")

# Step 2: Add edges from FEATURES (relationship pointers in v4)
edges_added = 0
dangling_pointers = 0

for (source, rid), record in record_by_id.items():
    from_node = f"{source}_{rid}"

    for feat in _get_features(record):
        pointer_domain = feat.get("REL_POINTER_DOMAIN")
        pointer_key = feat.get("REL_POINTER_KEY")
        role = feat.get("REL_POINTER_ROLE", "")

        # Skip non-relationship features and anchor-only entries
        if not pointer_domain or not pointer_key or not role:
            continue

        # Resolve the target record
        target_id = record_id_by_domain_key.get((pointer_domain, pointer_key))
        if not target_id:
            dangling_pointers += 1
            continue

        to_node = f"{target_id[0]}_{target_id[1]}"

        # Parse ownership percentage from role strings like "shareholding 75% 100%"
        ownership_pct = ""
        role_clean = role
        if "%" in role:
            parts = role.split()
            role_clean = parts[0]
            pct_parts = [p for p in parts[1:] if "%" in p]
            if len(pct_parts) == 2:
                ownership_pct = f"{pct_parts[0]}-{pct_parts[1]}"
            elif pct_parts:
                ownership_pct = pct_parts[0]

        G.add_edge(
            from_node,
            to_node,
            role=role_clean,
            role_raw=role,
            ownership_pct=ownership_pct,
            from_date=feat.get("REL_POINTER_FROM_DATE", ""),
            thru_date=feat.get("REL_POINTER_THRU_DATE", ""),
            source_domain=pointer_domain,
        )
        edges_added += 1

print(f"Edges added: {edges_added}")
print(f"Dangling pointers (target not in dataset): {dangling_pointers}")

# Step 3: Graph statistics
print(f"\nRaw Data Graph Statistics:")
print(f"  Total nodes: {G.number_of_nodes()}")
print(f"  Total edges: {G.number_of_edges()}")
print(f"  Connected components (undirected): {nx.number_connected_components(G.to_undirected())}")

# Count by data source
os_nodes = sum(1 for _, d in G.nodes(data=True) if d["data_source"] == "OPEN-SANCTIONS")
oo_nodes = sum(1 for _, d in G.nodes(data=True) if d["data_source"] == "OPEN-OWNERSHIP")
print(f"  Open Sanctions nodes: {os_nodes}")
print(f"  Open Ownership nodes: {oo_nodes}")

# Count by type
persons = sum(1 for _, d in G.nodes(data=True) if d["record_type"] == "PERSON")
orgs = sum(1 for _, d in G.nodes(data=True) if d["record_type"] == "ORGANIZATION")
print(f"  Person nodes: {persons}")
print(f"  Organization nodes: {orgs}")

# Count by edge role
from collections import Counter
role_counts = Counter(d["role"] for _, _, d in G.edges(data=True))
print(f"\nEdge types:")
for role, count in role_counts.most_common():
    print(f"  {role}: {count}")

Nodes added: 282
Edges added: 396
Dangling pointers (target not in dataset): 205

Raw Data Graph Statistics:
  Total nodes: 282
  Total edges: 250
  Connected components (undirected): 118
  Open Sanctions nodes: 24
  Open Ownership nodes: 258
  Person nodes: 108
  Organization nodes: 174

Edge types:
  other_influence_or_control: 97
  voting_rights: 60
  appointment_of_board: 41
  shareholding: 37
  Family: 5
  УчрФЛ: 4
  Controlled by: 3
  Directorship: 2
  УчрЮЛРос: 1


In [5]:
# Show the disconnection between data sources
G_undirected = G.to_undirected()
components = list(nx.connected_components(G_undirected))

# Classify components by data source
pure_os = 0
pure_oo = 0
mixed = 0
for comp in components:
    sources = set()
    for node in comp:
        sources.add(G.nodes[node]["data_source"])
    if sources == {"OPEN-SANCTIONS"}:
        pure_os += 1
    elif sources == {"OPEN-OWNERSHIP"}:
        pure_oo += 1
    else:
        mixed += 1

print("Connected components by data source:")
print(f"  Pure Open Sanctions: {pure_os}")
print(f"  Pure Open Ownership: {pure_oo}")
print(f"  Mixed (cross-source): {mixed}")
print(f"\nThis confirms the two datasets are COMPLETELY DISCONNECTED")
print("without entity resolution — no component spans both sources.")

# Show largest components
comp_sizes = sorted([len(c) for c in components], reverse=True)
print(f"\nLargest 10 components by size: {comp_sizes[:10]}")
print(f"Isolated nodes (size-1 components): {comp_sizes.count(1)}")

Connected components by data source:
  Pure Open Sanctions: 11
  Pure Open Ownership: 107
  Mixed (cross-source): 0

This confirms the two datasets are COMPLETELY DISCONNECTED
without entity resolution — no component spans both sources.

Largest 10 components by size: [26, 13, 11, 9, 9, 8, 6, 5, 5, 5]
Isolated nodes (size-1 components): 56


## Visualize the Raw Data Graph

Render the graph using PyVis with the same visual conventions as notebook 04's combined graph. Compare this visualization with notebook 04 to see the structural differences:

**Node styling:**
- **Orange** = Person, **Blue** = Organization, **Gray** = Unknown
- **Green dots** = Open Ownership records, **Red dots** = Open Sanctions records
- Node shape distinguishes data source: **dot** for all records (no entity layer here)

**Edge styling:**
- **Red** = Open Sanctions relationships (Directorship, Family)
- **Green** = Open Ownership relationships (Shareholding, Voting Rights, Board Appointment)

**Key visual difference from notebook 04:** There is no entity layer (large triangles/stars/diamonds) and no gray dashed resolution edges. Every node is a raw record, and the Open Sanctions and Open Ownership clusters float as **completely separate islands**.

In [6]:
net = Network(
    height="1200px",
    width="100%",
    bgcolor="#ffffff",
    font_color="#000000",
    notebook=True,
)

# Physics tuned for cluster spreading (same as nb04 combined graph)
net.repulsion(
    node_distance=250,
    central_gravity=0.1,
    spring_length=300,
    spring_strength=0.02,
    damping=0.3,
)

# Node color by data source (matching nb04 record-node colors)
source_color_map = {
    "OPEN-OWNERSHIP": "#2ca02c",   # Green
    "OPEN-SANCTIONS": "#e74c3c",   # Red
}

# Type-based color (matching nb04 entity-node colors)
type_color_map = {
    "PERSON": "#ff7f0e",           # Orange
    "ORGANIZATION": "#1f77b4",     # Blue
    "UNKNOWN": "#7f7f7f",          # Gray
}

# Edge color by source domain
edge_color_map = {
    "OPEN-SANCTIONS": "#e74c3c",   # Red
    "OOR": "#2ca02c",              # Green
}

# Add nodes
for node_id, nd in G.nodes(data=True):
    name = nd.get("name", "Unknown")
    record_type = nd.get("record_type", "UNKNOWN")
    data_source = nd.get("data_source", "UNKNOWN")
    risks = nd.get("risks", [])

    # Color by type, border by source
    bg_color = type_color_map.get(record_type, "#7f7f7f")
    border_color = source_color_map.get(data_source, "#95a5a6")

    # Size: slightly larger for nodes with risk flags
    size = 20 if risks else 15

    # Tooltip
    tooltip_parts = [
        name,
        f"Type: {record_type}",
        f"Source: {data_source}",
        f"Record ID: {nd.get('record_id', '')}",
    ]
    if risks:
        tooltip_parts.append(f"Risks: {', '.join(risks)}")
    tooltip = "\n".join(tooltip_parts)

    display_label = name[:25] + "..." if len(name) > 25 else name

    net.add_node(
        node_id,
        label=display_label,
        title=tooltip,
        color={"background": bg_color, "border": border_color},
        size=size,
        shape="dot",
        borderWidth=3,
    )

# Add edges
for source_node, target_node, edge_data in G.edges(data=True):
    role = edge_data.get("role", "related")
    role_raw = edge_data.get("role_raw", role)
    domain = edge_data.get("source_domain", "")
    edge_color = edge_color_map.get(domain, "#95a5a6")

    # Label: first word of role
    label = role.split()[0] if role else "related"

    net.add_edge(
        source_node,
        target_node,
        label=label,
        title=role_raw,
        color=edge_color,
        width=2,
        font={"size": 9, "color": edge_color},
        arrows="to",
        smooth={"type": "curvedCW", "roundness": 0.1},
    )

# Render inline — generate HTML and inject physics auto-disable script
html_content = net.generate_html()

physics_auto_off = """
<script type="text/javascript">
  setTimeout(function() {
    network.setOptions({ physics: false });
    console.log("Physics auto-disabled after 15 seconds");
  }, 15000);
</script>
"""
html_content = html_content.replace("</body>", physics_auto_off + "</body>")

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print(f"Compare with notebook 04's combined graph (478 nodes, 774 edges)")
display(HTML(html_content))

Nodes: 282, Edges: 250
Compare with notebook 04's combined graph (478 nodes, 774 edges)


## Build Text Descriptions and Vectorize

Convert each raw record into a human-readable text description (same approach as notebook 05), then embed with `all-MiniLM-L6-v2` and store in LanceDB.

In [7]:
def record_to_text(record: dict) -> str:
    """Convert a raw JSON record into a meaningful human-readable string."""
    parts = []
    features = _get_features(record)

    name = get_name(record)
    record_type = get_record_type(record)
    source = record.get("DATA_SOURCE", "UNKNOWN")

    parts.append(f"{name} is a {record_type.lower()} from the {source} dataset.")

    # Extract fields from FEATURES
    addresses = []
    dob = None
    nationality = None
    reg_country = None
    reg_date = None
    identifiers = []
    relationships = []

    for feat in features:
        if feat.get("ADDR_FULL"):
            addresses.append(feat["ADDR_FULL"])
        if feat.get("DATE_OF_BIRTH"):
            dob = feat["DATE_OF_BIRTH"]
        if feat.get("NATIONALITY"):
            nationality = feat["NATIONALITY"]
        if feat.get("REGISTRATION_COUNTRY"):
            reg_country = feat["REGISTRATION_COUNTRY"]
        if feat.get("REGISTRATION_DATE"):
            reg_date = feat["REGISTRATION_DATE"]
        # Identifiers
        id_type = feat.get("OTHER_ID_TYPE") or feat.get("NATIONAL_ID_TYPE") or ""
        id_num = feat.get("OTHER_ID_NUMBER") or feat.get("NATIONAL_ID_NUMBER") or ""
        if id_type and id_num:
            identifiers.append(f"{id_type}: {id_num}")
        elif id_num:
            identifiers.append(id_num)
        # Relationships — resolve pointer to a name where possible
        if feat.get("REL_POINTER_ROLE"):
            role = feat["REL_POINTER_ROLE"]
            domain = feat.get("REL_POINTER_DOMAIN", "")
            key = feat.get("REL_POINTER_KEY", "")

            target = record_id_by_domain_key.get((domain, key))
            if target:
                target_record = record_by_id.get(target)
                target_name = get_name(target_record) if target_record else key
            else:
                target_name = key

            rel_str = f"{role} with {target_name}"
            from_date = feat.get("REL_POINTER_FROM_DATE", "")
            thru_date = feat.get("REL_POINTER_THRU_DATE", "")
            if from_date:
                rel_str += f" from {from_date}"
            if thru_date:
                rel_str += f" to {thru_date}"
            relationships.append(rel_str)

    if addresses:
        parts.append(f"Address: {'; '.join(addresses)}.")
    if dob:
        parts.append(f"Date of birth: {dob}.")
    if reg_date:
        parts.append(f"Registration date: {reg_date}.")
    # Top-level dates (Open Ownership)
    if record.get("REGISTRATION_DATE"):
        parts.append(f"Registration date: {record['REGISTRATION_DATE']}.")
    if record.get("dissolutionDate"):
        parts.append(f"Dissolution date: {record['dissolutionDate']}.")
    if nationality:
        parts.append(f"Nationality: {nationality.upper()}.")
    if reg_country:
        parts.append(f"Registered in: {reg_country.upper()}.")

    # Risks (top-level fields in v4)
    risk_topics = record.get("risk_topics", "")
    if risk_topics:
        parts.append(f"Risk flags: {risk_topics}.")

    if identifiers:
        parts.append(f"Identifiers: {', '.join(identifiers)}.")
    if relationships:
        parts.append(f"Relationships: {'; '.join(relationships)}.")

    return " ".join(parts)


# Test with sample records
print("=== Sample Open Sanctions record ===")
print(record_to_text(sanctions_records[0]))
print()
print("=== Sample Open Ownership record ===")
print(record_to_text(ownership_records[0]))

=== Sample Open Sanctions record ===
Abassin BADSHAH is a person from the OPEN-SANCTIONS dataset. Address: 31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL. Date of birth: 1985-05-12. Nationality: GB. Risk flags: corp.disqual. Identifiers: OPEN-SANCTIONS: NK-25vyVFzt8vdJGgAXMRTwTJ. Relationships: Directorship with BARLLOWS SERVICES LTD; Directorship with LMAR (GB) LTD.

=== Sample Open Ownership record ===
GOLD WYNN UK HOLDINGS LIMITED is a organization from the OPEN-OWNERSHIP dataset. Address: C/O Fladgate Llp, 16 Great Queen Street, London, WC2B 5DG. Registration date: 2020-03-18. Registered in: GB. Identifiers: GB-COH: 12524623. Relationships: shareholding 75% 100% with Wynn Weinzweig Uk Holdings Llp; voting_rights 75% 100% with Wynn Weinzweig Uk Holdings Llp; appointment_of_board with Wynn Weinzweig Uk Holdings Llp.


In [8]:
print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding model loaded (dimension: {embedding_model.get_sentence_embedding_dimension()})")

# Build rows for LanceDB
print(f"\nProcessing {len(record_by_id)} records...")

rows = []
for (source, rid), record in record_by_id.items():
    text = record_to_text(record)
    name = get_name(record)
    node_id = f"{source}_{rid}"
    risk_topics = record.get("risk_topics", "")
    risks = [t.strip() for t in risk_topics.split(",") if t.strip()] if risk_topics else []

    rows.append({
        "node_id": node_id,
        "record_id": rid,
        "data_source": source,
        "record_type": get_record_type(record),
        "name": name,
        "text": text,
        "risks": "; ".join(risks) if risks else "",
    })

# Batch encode all texts
texts = [r["text"] for r in rows]
embeddings = embedding_model.encode(texts, show_progress_bar=True)

for i, row in enumerate(rows):
    row["vector"] = embeddings[i].tolist()

print(f"Embeddings created: {len(embeddings)} x {embeddings.shape[1]}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded (dimension: 384)

Processing 282 records...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embeddings created: 282 x 384


## Store in LanceDB

Clear any existing tables and store the vectorized records in a fresh `raw_graph_records` table.

In [9]:
db = lancedb.connect(LANCEDB_PATH)

# Drop existing tables to start fresh
for name in db.table_names():
    db.drop_table(name)
    print(f"Dropped table: {name}")

table = db.create_table("raw_graph_records", data=rows)

print(f"\nLanceDB table 'raw_graph_records' created with {table.count_rows()} rows")
print(f"\nSample entries:")
sample = table.to_pandas().head(5)
print(sample[["node_id", "data_source", "record_type", "name", "risks"]].to_string())

Dropped table: raw_records

LanceDB table 'raw_graph_records' created with 282 rows

Sample entries:
                                    node_id     data_source   record_type                               name               risks
0  OPEN-SANCTIONS_NK-25vyVFzt8vdJGgAXMRTwTJ  OPEN-SANCTIONS        PERSON                    Abassin BADSHAH        corp.disqual
1  OPEN-SANCTIONS_NK-3p3mmVWmjwVtTfKchz4kNE  OPEN-SANCTIONS  ORGANIZATION                      LMAR (GB) LTD                    
2  OPEN-SANCTIONS_NK-auyPsLrBzRoxjCRWgjBvas  OPEN-SANCTIONS  ORGANIZATION            WANDLE HOLDINGS LIMITED     sanction.linked
3  OPEN-SANCTIONS_NK-cf4Q3KcmUnQbt8Cy7iTtwK  OPEN-SANCTIONS  ORGANIZATION  POLYUS GOLD INTERNATIONAL LIMITED     sanction.linked
4  OPEN-SANCTIONS_NK-dNNN56A4ApVfUFvfzniLCF  OPEN-SANCTIONS        PERSON          Firuza Nazimovna Kerimova  role.rca; sanction


/tmp/ipykernel_2840/4269279080.py:4: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  for name in db.table_names():


## Set Up LLM Clients and Graph RAG Query Function

Configure the Anthropic and OpenAI clients, then define the RAG pipeline: vector search in LanceDB, neighbor expansion via the raw data graph, context assembly, and LLM query.

The key difference from notebook 05's flat RAG: after vector search finds the top-k records, we **expand to graph neighbors** to pull in related entities (directors, shareholders, subsidiaries). This gives the LLM richer context — but without entity resolution, the expansion is limited to within each data source.

In [10]:
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Anthropic client ready:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("OpenAI client ready:", bool(os.getenv("OPENAI_API_KEY")))

Anthropic client ready: True
OpenAI client ready: True


In [11]:
SYSTEM_PROMPT = (
    "Answer questions about a corporate ownership and sanctions dataset. "
    "You have access to raw source records from Open Sanctions and Open Ownership, "
    "organized in a graph with relationship edges (directorships, shareholdings, "
    "voting rights, board appointments, family ties). "
    "IMPORTANT: These records have NOT been entity-resolved — the same real-world "
    "entity may appear as separate records in different data sources with NO link "
    "between them. The Open Sanctions and Open Ownership subgraphs are completely "
    "disconnected. If you suspect two records refer to the same entity across sources, "
    "note this as a possibility but be clear that it is not confirmed by the data."
)


def ask_graph_rag(question: str, provider: str = "anthropic", top_k: int = 10) -> None:
    """
    Graph-augmented RAG over raw (non-entity-resolved) records.

    1. Vector search to find top-k relevant records
    2. Expand to graph neighbors (1 hop) for relationship context
    3. Build context with record details and relationship info
    4. Query the LLM

    Parameters
    ----------
    question : str
        The user's question.
    provider : str
        LLM backend — "anthropic" (Claude) or "openai" (GPT).
    top_k : int
        Number of records to retrieve via vector search.
    """
    if provider not in ("anthropic", "openai"):
        raise ValueError(f"provider must be 'anthropic' or 'openai', got '{provider}'")

    print(f"\nQuestion: {question}")
    print(f"Provider: {provider}")
    print("=" * 70)

    # Step 1: Vector search
    question_embedding = embedding_model.encode(question).tolist()
    results = table.search(question_embedding).limit(top_k).to_list()
    print(f"Vector search returned {len(results)} records")

    # Step 2: Expand to graph neighbors
    seed_nodes = set()
    for r in results:
        seed_nodes.add(r["node_id"])

    expanded_nodes = set(seed_nodes)
    for node_id in seed_nodes:
        if node_id in G:
            # Follow edges in both directions (successors + predecessors)
            for neighbor in list(G.successors(node_id))[:8]:
                expanded_nodes.add(neighbor)
            for neighbor in list(G.predecessors(node_id))[:8]:
                expanded_nodes.add(neighbor)

    print(f"Expanded to {len(expanded_nodes)} records (including graph neighbors)")

    # Step 3: Build context
    context_parts = ["RAW DATA GRAPH — RECORDS AND RELATIONSHIPS:\n"]

    for node_id in expanded_nodes:
        if node_id not in G:
            continue

        nd = G.nodes[node_id]
        name = nd.get("name", "Unknown")
        record_type = nd.get("record_type", "UNKNOWN")
        source = nd.get("data_source", "UNKNOWN")
        risks = nd.get("risks", [])
        is_seed = node_id in seed_nodes

        marker = "[DIRECT MATCH] " if is_seed else "[GRAPH NEIGHBOR] "
        context_parts.append(f"{marker}{name} ({record_type}, {source})")

        if risks:
            context_parts.append(f"  Risk flags: {', '.join(risks)}")

        # Show outgoing relationships
        out_rels = []
        for _, target, edge_data in G.out_edges(node_id, data=True):
            if target in G:
                target_name = G.nodes[target].get("name", "Unknown")
                role = edge_data.get("role_raw", edge_data.get("role", "related"))
                rel_str = f"{role} → {target_name}"
                if edge_data.get("from_date"):
                    rel_str += f" (from {edge_data['from_date']}"
                    if edge_data.get("thru_date"):
                        rel_str += f" to {edge_data['thru_date']}"
                    rel_str += ")"
                out_rels.append(rel_str)

        # Show incoming relationships
        in_rels = []
        for source_node, _, edge_data in G.in_edges(node_id, data=True):
            if source_node in G:
                source_name = G.nodes[source_node].get("name", "Unknown")
                role = edge_data.get("role_raw", edge_data.get("role", "related"))
                rel_str = f"{source_name} → {role}"
                in_rels.append(rel_str)

        if out_rels:
            context_parts.append(f"  Outgoing: {'; '.join(out_rels[:5])}")
        if in_rels:
            context_parts.append(f"  Incoming: {'; '.join(in_rels[:5])}")

        context_parts.append("")

    context = "\n".join(context_parts)

    # Step 4: Query LLM
    print("Querying LLM...")
    user_message = f"Context:\n{context}\n\nQuestion: {question}"

    if provider == "anthropic":
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )
        answer = response.content[0].text
    else:
        response = openai_client.chat.completions.create(
            model="gpt-5.4-nano",
            max_tokens=2048,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
        )
        answer = response.choices[0].message.content

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)
    print(answer)
    print("=" * 70)

## Interactive Query Session

Ask questions against the raw data graph. The system uses vector search + graph neighbor expansion, but **without entity resolution** the two data sources remain disconnected.

Change `provider` below to `"openai"` to use GPT instead of Claude.

In [ ]:
provider = "anthropic"  # change to "openai" to use GPT-5.4 nano

print("Graph RAG on Raw Data (No Entity Resolution) - Interactive Session")
print("=" * 70)
print("Ask questions about the corporate ownership and sanctions data.")
print("This uses GRAPH-AUGMENTED RAG on RAW records — no entity resolution.")
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Current LLM provider: {provider}")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()

    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    if not question:
        continue

    try:
        ask_graph_rag(question, provider=provider)
        print()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

Graph RAG on Raw Data (No Entity Resolution) - Interactive Session
Ask questions about the corporate ownership and sanctions data.
This uses GRAPH-AUGMENTED RAG on RAW records — no entity resolution.
Graph: 282 nodes, 250 edges
Current LLM provider: anthropic
Type 'quit' to exit.



Your question:  tell me about said kerimov



Question: tell me about said kerimov
Provider: anthropic
Vector search returned 10 records
Expanded to 16 records (including graph neighbors)
Querying LLM...

ANSWER
# Said Kerimov - Profile

Based on the available data, here's what we know about Said Kerimov:

## **Sanctions Status**
- **Sanctioned individual** with RCA (relatives and close associates) designation
- Connected to the sanctioned Russian oligarch **Suleyman Abusaidovich KERIMOV** through a family relationship

## **Family Connections**
Said Kerimov has a documented **family relationship** with:
- **Suleyman Abusaidovich KERIMOV** - a sanctioned Russian oligarch and politically exposed person (PEP)

Other family members of Suleyman Kerimov who are also sanctioned include:
- Firuza Nazimovna Kerimova
- Amina Suleymanovna Kerimova  
- Gulnara Suleimanova Kerimova

## **Russian Business Interests** (Open Sanctions data)
Said Kerimov has controlling interests (УчрФЛ - "founder/participant" relationships) in several Russian e

Your question:  how do we know Firuza Nazimovna Kerimova is a family member of said kerimov?



Question: how do we know Firuza Nazimovna Kerimova is a family member of said kerimov?
Provider: anthropic
Vector search returned 10 records
Expanded to 19 records (including graph neighbors)
Querying LLM...

ANSWER
Based on the raw data graph, we can trace the family relationship between Firuza Nazimovna Kerimova and Said Kerimov through **Suleyman Abusaidovich KERIMOV** as the connecting link:

## The Connection Chain:

1. **Firuza Nazimovna Kerimova** (Open Sanctions) has an outgoing "Family" relationship → **Suleyman Abusaidovich KERIMOV**

2. **Said Kerimov** (Open Sanctions) has an outgoing "Family" relationship → **Suleyman Abusaidovich KERIMOV**

3. **Suleyman Abusaidovich KERIMOV** is flagged as an oligarch, PEP (Politically Exposed Person), person of interest, and is under sanctions

## What This Tells Us:

Both Firuza and Said have family relationships to the same person (Suleyman Abusaidovich KERIMOV), making them family members **through** him. The data doesn't specify th